# **TAREA CALIFICADA 1**
**NOMBRE:** Alessandra Marcela Marocho Pacheco

### **DESCRIPCIÓN DEL PROYECTO**

**Descripción Técnica del Proyecto**

El presente trabajo consiste en el desarrollo de un "Research Copilot", un asistente conversacional de Inteligencia Artificial basado en la arquitectura RAG (Retrieval-Augmented Generation). Esta herramienta ha sido diseñada para procesar, consultar y extraer información de forma eficiente a partir de una colección estructurada de 20 artículos académicos.


**Contexto y Objeto de Estudio**

Los documentos integrados en el sistema conforman la revisión de literatura de mi proyecto de tesis de licenciatura en Relaciones Internacionales, titulada *"Más allá del comercio: la centralidad ASEAN en las dinámicas de regionalismo en el Este asiático durante en el proceso de negociaciones del RCEP (2012-2022)".*

Dicha investigación analiza las estrategias y mecanismos diplomáticos empleados por la Asociación de Naciones de Asia Sudoriental (ASEAN) durante las negociaciones del tratado RCEP (2012-2022). El objetivo central es evaluar cómo la ASEAN busca preservar su centralidad y relevancia en el Este Asiático frente a la reconfiguración del orden regional y la emergencia de nuevas potencias.

In [1]:
#!pip install openai tiktoken pypdf chromadb langchain langchain-text-splitters
#!pip install PyMuPDF
#!pip install gradio

Ubicamos la carpeta en la que se encuentran los papers a utilizar:

In [2]:
import os
import re
import fitz
import tiktoken
import chromadb
import json
from openai import OpenAI
from getpass import getpass

papers_folder = "A:/Github/research_copilot_asean/papers/"

if not os.path.exists(papers_folder):
    os.makedirs(papers_folder)
    print(f"Carpeta '{papers_folder}' verificada/creada.")

In [3]:
# 1. Configuración de credenciales de OpenAI
print("Configuración de credenciales:")
api_key = getpass("Ingresa tu OpenAI API Key: ")
client = OpenAI(api_key=api_key)

# 2. Definición de la carpeta de trabajo
papers_folder = "papers/"

Configuración de credenciales:


### **CREACIÓN DEL JSON**

In [4]:
json_path = os.path.join(papers_folder, "paper_catalog.json")

# Diccionario completo con tus 20 papers
catalog_data = {
    "papers": [
        {
            "id": "paper_001",
            "title": "From APEC to mega-regionals: the evolution of the Asia-Pacific trade architecture",
            "authors": ["Mireya Solís", "Jeffrey D. Wilson"],
            "year": 2017,
            "venue": "The Pacific Review",
            "doi": "10.1080/09512748.2017.1305438",
            "filename": "From APEC to mega-regionals the evolution of the Asia-Pacific trade architecture.pdf",
            "topics": ["ASEAN", "RCEP", "regionalismo", "Asia-Pacífico"],
            "abstract": "Of the recent transformations in the political economy of the Asia-Pacific, one of the most dramatic has been to the region's trade architecture. For many years, Asian government were committed trade multilateralists: pursuing liberalisation either globally through the GATT, or regionally via APEC's model of open regionalism. Underpinned by US and Japanese leadership, this system provided the foundation for the export-driven Asian economic miracle. But since the early twenty-first century, the system has been rapidly transformed. The proliferation of preferential trade agreements has threatened to undermine the cohesiveness of regional trade arrangements. The emergence of WTO-Plus style liberalisation, emphasising services, investment and intellectual property, marks the maturation of a system previously focussed on tariff reduction and manufacturing exports. Since 2011, competition between two ‘mega-regional’ proposals– the Trans-Pacific Partnership and the Regional Comprehensive Economic Partnership– is also indicative of new splits which cut across traditional developmental divides. Growing geopolitical rivalry between the US and China has also raised question of who will lead the next round of liberalisation in the region. Exploring these new trends, this paper argues the trade architecture of the Asia-Pacific is entering is becoming more contested and fragmented, with major implications for economic regionalism in coming years."
        },
        {
            "id": "paper_002",
            "title": "RCEP from the middle powers' Perspective",
            "authors": ["Fukunari Kimura"],
            "year": 2021,
            "venue": "China Economic Journal",
            "doi": "10.1080/17538963.2021.1933059",
            "filename": "RCEP from the middle powers Perspective .pdf",
            "topics": ["ASEAN", "RCEP", "regionalismo", "middle powers"],
            "abstract": "East Asian countries signed the Regional Comprehensive Economic Partnership Agreement (RCEP) in November 2020. This paper demonstrates the importance of ASEAN centrality in East Asian economic integration and makes a preliminary assessment of the agreement in terms of the four expected roles: liberalization, rule making, reducing policy risks, and forming a pro-trade middle power coalition. The comparison with the Comprehensive and Progressive Agreement for Trans-Pacific Partnership (CPTPP) reveals the strengths and weaknesses of RCEP. The paper emphasizes the importance of dynamic aspects of mega-FTAs after being in effect and claims that RCEP must be further developed as an evolving agreement."
        },
        {
            "id": "paper_003",
            "title": "ASEAN's inheritance: the regionalization of Southeast Asia, 1941-61",
            "authors": ["Philip Charrier"],
            "year": 2001,
            "venue": "The Pacific Review",
            "doi": "10.1080/09512740110064811",
            "filename": "ASEAN s inheritance  the regionalization of Southeast Asia  1941 61.pdf",
            "topics": ["ASEAN", "Southeast Asia", "regionalismo histórico"],
            "abstract": "This article explores the establishment of Southeast Asia as a regional space in the international system before the advent of ASEAN..."
        },
        {
            "id": "paper_004",
            "title": "Asean's inclusive regionalism: ambitious at three levels",
            "authors": ["Astanah Abdul Aziz", "Anthony Milner"],
            "year": 2024,
            "venue": "Australian Journal of International Affairs",
            "doi": "10.1080/10357718.2024.2313484",
            "filename": "ASEAN inclusive regionalism ambitious at three levels .pdf",
            "topics": ["ASEAN", "inclusive regionalism"],
            "abstract": "This commentary examines ASEAN’s 'inclusive regionalism' as a strategic framework to maintain ASEAN Centrality amid Indo-Pacific geopolitical rivalries. The authors argue that this approach operates across three levels: internal cohesion, regional institutional binding through mechanisms like RCEP, and the global promotion of the 'ASEAN Way.' The study evaluates whether this multi-layered inclusivity remains a viable strategy for stability or if it faces over-ambition in an increasingly polarized international order."
        },
        {
            "id": "paper_005",
            "title": "Investigating merchandise trade structure in the RCEP region from the perspective of regional integration",
            "authors": ["CHEN Xiaoqiang", "YUAN Lihua", "SONG Changqing"],
            "year": 2023,
            "venue": "Journal of Geographical Sciences",
            "doi": "10.1007/s11442-023-2125-7",
            "filename": "Investigating merchandise trade structure in the RCEP region from the perspective of regional integration.pdf",
            "topics": ["RCEP", "regional integration", "merchandise trade"],
            "abstract": "The Regional Comprehensive Economic Partnership (RCEP) was formally signed by the Association of Southeast Asian Nations (ASEAN) countries..."
        },
        {
            "id": "paper_006",
            "title": "Hedging via Institutions: ASEAN-led Multilateralism in the Age of the Indo-Pacific",
            "authors": ["Cheng-Chwee Kuik"],
            "year": 2022,
            "venue": "Asian Journal of Peacebuilding",
            "doi": "10.18588/202211.00a319",
            "filename": "Hedging via Institutions ASEAN-led Multilateralism in the Age of the Indo-Pacific.pdf",
            "topics": ["ASEAN", "multilateralism", "hedging", "Indo-Pacific"],
            "abstract": "This article unpacks the dynamics of group hedging in international relations by examining the Southeast Asian states' collective efforts..."
        },
        {
            "id": "paper_007",
            "title": "Assessing the Impact of the Regional Comprehensive Economic Partnership on ASEAN Trade",
            "authors": ["Sithanonxay Suvannaphakdy"],
            "year": 2021,
            "venue": "Journal of Southeast Asian Economies",
            "doi": "10.2307/27035509",
            "filename": "Assessing the Impact of the Regional Comprehensive Economic Partnership on ASEAN Trade.pdf",
            "topics": ["RCEP", "ASEAN", "trade impact"],
            "abstract": "The Regional Comprehensive Economic Partnership (RCEP) is a regionwide free trade agreement (FTA) linking the ten ASEAN economies to their “+5” partners, namely Australia, China, Japan, New Zealand and South Korea. It covers both trade and non-trade related issues ranging from rules of origin and trade facilitation to intellectual property rights and investment. This study examines the likely impact of RCEP on trade alone, taking into account the fact that all its members are already participants in a number of other FTAs. Using latest FTA data from the WTO on imports and exports, this study reveals that tariff reduction under RCEP will erode ASEAN’s trade preferences provided by existing FTA partners, while reallocating import sources of ASEAN countries towards more efficient RCEP partners."
        },
        {
            "id": "paper_008",
            "title": "Mega-Regional Trade Deals in the Asia-Pacific: Choosing Between the TPP and RCEP?",
            "authors": ["Jeffrey D. Wilson"],
            "year": 2015,
            "venue": "Journal of Contemporary Asia",
            "doi": "10.1080/00472336.2014.956138",
            "filename": "Mega-Regional Trade Deals in the Asia-Pacific Choosing Between the TPP and RCEP.pdf",
            "topics": ["TPP", "RCEP", "Asia-Pacific", "mega-regional trade deals"],
            "abstract": "The emergence of “mega-regional” trade agreements has recently become the most significant trade policy issue in the Asia-Pacific. Since 2010, governments in the region have launched negotiations for two new trade agreements: the United States-led Trans-Pacific Partnership (TPP) and the ASEAN-led Regional Comprehensive Economic Partnership (RCEP). Differentiated by their membership, scope and level of ambition, the TPP and RCEP embody competing visions for how the Asia-Pacific trade system should evolve, and regional governments must now make choices over which initiative better serves their economic and political interests. This article explores the trade policy choice posed by these mega-regional trade negotiations, reviewing the evolution of the Asia-Pacific trade system, the recent emergence of the TPP and RCEP, and the competitive dynamics inherent in the development of the two proposals. It argues that four key considerations (trade policy ambition, the role of ASEAN, US-China geopolitical rivalry and defensive concerns) will be of key importance in informing regional governments’ decisions as the TPP and RCEP move towards completion in 2015."
        },
        {
            "id": "paper_009",
            "title": "RCEP: a strategic opportunity for multilateralism",
            "authors": ["Peter Drysdale", "Shiro Armstrong"],
            "year": 2021,
            "venue": "China Economic Journal",
            "doi": "10.1080/17538963.2021.1937092",
            "filename": "RCEP a strategic opportunity for multilateralism.pdf",
            "topics": ["RCEP", "multilateralism", "strategic opportunity"],
            "abstract": "East Asia’s Regional Comprehensive Economic Partnership (RCEP) agreement was concluded in a time of heightened uncertainty in the global economy and in the middle of the largest economic downturn in almost a century from a pandemic-induced global recession. The agreement consolidated the 10 member ASEAN’s free trade agreements with Australia and New Zealand, and the three northeast Asian economic powers China, Japan and South Korea. RCEP's economic cooperation agenda incorporates ASEAN processes which go beyond helping countries to implement the agreement and has potential to expand cooperation to new areas. This economic cooperation process makes RCEP a living agreement that can serve the needs of members as they evolve. It can also be used to embrace non-RCEP members, especially India, around par ticular agendas. The agreement is an important opportunity for China to use the RCEP framework to trial reforms and demonstrate its commitment to broader international multilateral liberalisation and economic cooperation."
        },
        {
            "id": "paper_010",
            "title": "Towards a New ASEAN Regionalism: Navigating the Outlook on Indo-Pacific in Post-RCEP Beyond 2020",
            "authors": ["Hino Samuel Jose", "Hree Dharma Santhi Putri Samudra"],
            "year": 2022,
            "venue": "Insignia Journal of International Relations",
            "doi": "10.20884/1.ins.2022.9.1.4636",
            "filename": "Towards a New ASEAN Regionalism.pdf",
            "topics": ["ASEAN", "RCEP", "Indo-Pacific", "new regionalism"],
            "abstract": "The adoption of the Regional Comprehensive Economic Partnership (RCEP) has brought the Asia Pacific region into a new paradigm of ASEAN regionalism..."
        },
        {
            "id": "paper_011",
            "title": "Building ASEAN Identity Through Regional Diplomacy",
            "authors": ["Elaine Tan"],
            "year": 2022,
            "venue": "Global Political Transitions",
            "doi": "10.1007/978-981-16-7007-7_11",
            "filename": "Building ASEAN Identity Through Regional Diplomacy.pdf",
            "topics": ["ASEAN", "identity", "regional diplomacy", "centrality"],
            "abstract": "When the historic Bangkok Declaration was signed on 8 August 1967..."
        },
        {
            "id": "paper_012",
            "title": "Beyond the 'new' regionalism",
            "authors": ["Björn Hettne"],
            "year": 2005,
            "venue": "New Political Economy",
            "doi": "10.1080/13563460500344484",
            "filename": "Beyond the ‘New’ Regionalism.pdf",
            "topics": ["regionalism", "political economy"],
            "abstract": "This article critiques the 'new regionalism' wave, arguing for a more comprehensive understanding of regional integration in a globalized world. It explores the transition from state-led economic blocs to multidimensional regional societies, emphasizing the role of regionality in providing stability and a 'new multilateralism' to counter global uncertainties."
        },
        {
            "id": "paper_013",
            "title": "Doomed by Dialogue: Will ASEAN Survive Great Power Rivalry in Asia?",
            "authors": ["Amitav Acharya"],
            "year": 2018,
            "venue": "International Relations",
            "doi": "10.1007/978-981-10-3171-7_6",
            "filename": "Doomed by Dialogue Will ASEAN Survive Great Power Rivalry in Asia.pdf",
            "topics": ["ASEAN", "great power rivalry", "Asia-Pacific", "Indo-Pacific"],
            "abstract": "Pundits and policymakers increasingly see the changing great power politics in Asia..."
        },
        {
            "id": "paper_014",
            "title": "Thirty years of ASEAN: achievements and challenges",
            "authors": ["Jörn Dosch", "Manfred Mols"],
            "year": 1998,
            "venue": "The Pacific Review",
            "doi": "10.1080/09512749808719251",
            "filename": "Thirty years of ASEAN Achievements and challenges.pdf",
            "topics": ["ASEAN", "regionalism", "Southeast Asia"],
            "abstract": "This article first takes stock of the experiences, successes and shortcomings of Southeast Asian regionalism..."
        },
        {
            "id": "paper_015",
            "title": "The Emergence and Proliferation of Free Trade Agreements in East Asia",
            "authors": ["Shujiro Urata"],
            "year": 2004,
            "venue": "Japanese Economy",
            "doi": "10.1080/2329194X.2004.11045188",
            "filename": "The Emergence and Proliferation of Free Trade Agreements in East Asia.pdf",
            "topics": ["East Asia", "Free Trade Agreements", "proliferation"],
            "abstract": "This article analyzes the emergence and proliferation of free trade agreements (FTAs) in East Asia, examining the factors that have driven their growth and the implications for regional economic integration."
        },
        {
            "id": "paper_016",
            "title": "Trade Diversion and Creation Effect of Free Trade Agreements in ASEAN: Do Institutions Matter?",
            "authors": ["Abdulkareem Alhassan", "Cem Payaslioğlu"],
            "year": 2023,
            "venue": "Journal of the Knowledge Economy",
            "doi": "10.1007/s13132-023-01108-z",
            "filename": "Trade Diversion and Creation Effect of Free Trade Agreements in ASEAN Do Institutions Matter.pdf",
            "topics": ["ASEAN", "Free Trade Agreements", "trade diversion", "institutions"],
            "abstract": "There has been a proliferation of free trade agreements (FTAs) over the last three decades..."
        },
        {
            "id": "paper_017",
            "title": "China's rising assertiveness and the decline in the East Asian regionalism narrative",
            "authors": ["Andrew Yeo"],
            "year": 2020,
            "venue": "International Relations of the Asia-Pacific",
            "doi": "10.1093/irap/lcz013",
            "filename": "China’s rising assertiveness and the decline in the East Asian regionalism narrative.pdf",
            "topics": ["China", "East Asian regionalism", "assertiveness"],
            "abstract": "After a decade of vibrant scholarly and political discourse regarding the prospects of East Asian integration..."
        },
        {
            "id": "paper_018",
            "title": "Why Asian states cooperate in regional arrangements: Asian regionalism in comparative perspective",
            "authors": ["Diana Panke", "Jürgen Rüland"],
            "year": 2022,
            "venue": "International Relations",
            "doi": "10.1177/00471178211028970",
            "filename": "Why Asian states cooperate in regional arrangements Asian regionalism in comparative perspective.pdf",
            "topics": ["Asian regionalism", "regional cooperation", "comparative perspective"],
            "abstract": "Regional cooperation in Asia takes place in formal Regional Organizations (ROs) as well as in less formal Regional Fora (RF)..."
        },
        {
            "id": "paper_019",
            "title": "The ASEAN Economic Community and the RCEP in the world economy",
            "authors": ["Kazushi Shimizu"],
            "year": 2021,
            "venue": "Journal of Contemporary East Asia Studies",
            "doi": "10.1080/24761028.2021.1907881",
            "filename": "The ASEAN Economic Community and the RCEP in the world economy.pdf",
            "topics": ["ASEAN Economic Community", "RCEP", "world economy"],
            "abstract": "The Association of Southeast Asian Nations (ASEAN) has been leading economic integration within many structural changes in the world economy. ASEAN, established in 1967, has promoted regional eco nomic integration since 1976. It started to work toward realizing the ASEAN Free Trade Area (AFTA) in 1992 and the ASEAN Economic Community (AEC) in 2003. ASEAN finally established the AEC at the end of 2015. The AEC is the most developed and advanced economic integration in East Asia. ASEAN is deepening the AEC for the next goal, AEC 2025. ASEAN also led East Asian cooperation initiatives, including ASEAN+3 and ASEAN+6, and ASEAN+1 FTAs. ASEAN proposed the Regional Comprehensive Economic Partnership (RCEP) and led the RCEP negotiations. Currently, rising protectionism and the US–China trade friction has great negative impacts on ASEAN and East Asia. Furthermore, the outbreak of COVID-19 has done great damage to ASEAN and East Asia. ASEAN is responding to the COVID-19 pandemic and strengthening the AEC steadily amidst growing protectionism and COVID-19 pandemic. The RCEP agreement was finally signed in November 2020. The RCEP is the first East Asian mega FTA. The RCEP has great meaning in ASEAN and East Asia. ASEAN secured ASEAN centrality in East Asian economic integration. The AEC and the RCEP will become more important amidst rising protectionism, and during and in the post-pandemic era."
        },  
        {
            "id": "paper_020",
            "title": "China's Approach to Regional Free Trade Frameworks in the Asia Pacific: RCEP as a Prime Example of Economic Diplomacy?",
            "authors": ["David Groten"],
            "year": 2017,
            "venue": "Sicherheit und Frieden (S+F) / Security and Peace",
            "doi": "10.5771/0175-274X-2017-3-144",
            "filename": "China's Approach to Regional Free Trade Frameworks in the Asia Pacific RCEP as a Prime Example of Economic Diplomacy.pdf",
            "topics": ["China", "RCEP", "Economic Diplomacy", "Asia Pacific"],
            "abstract": "This article examines China's perception of two major free trade agreements (FTAs) in the Asia Pacific, the U.S.-led Transpacific Partnership Agreement (TPP) and the Regional Comprehensive Economic Partnership Agreement (RCEP) driven by the Association of Southeast Asian Nations (ASEAN) and The People's Republic of China. By means of scrutinizing nearly 800 publications by two leading Chinese Foreign Policy Think Tanks (FPTT) between 2010 and 2015, it is found that TPP is regarded by many experts as not merely an economic but also a strategic and political challenge. Accordingly, RCEP is commonly framed as an economic but also political and strategic response to TPP. In sum, it is revealed that FTAs are frequently perceived by Chinese FPTTs as vital tools of foreign economic diplomacy suitable to accomplish economic just as (geo-) political and strategic objectives."
        }
    ]
}

# Escribir el diccionario en un archivo JSON en la carpeta papers/
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(catalog_data, f, indent=4, ensure_ascii=False)

print(f"¡Catálogo generado exitosamente en: {json_path}!")

¡Catálogo generado exitosamente en: papers/paper_catalog.json!


### **PROCESAMIENTO DEL TEXTO**

In [5]:
def extract_text_from_pdf(pdf_path):
    """
    Extrae texto de un archivo PDF usando la librería PyMuPDF (fitz).
    Itera por cada página y concatena el contenido.
    """
    text = ""
    try:
        # Abrimos el documento PDF
        with fitz.open(pdf_path) as doc:
            for page in doc:
                # Extraemos el texto de la página actual
                text += page.get_text("text") + "\n"
    except Exception as e:
        print(f"Error al leer el archivo {pdf_path}: {e}")
    return text

def clean_text(text):
    """
    Realiza una limpieza básica: elimina múltiples espacios,
    saltos de línea excesivos y tabulaciones.
    """
    # Reemplaza cualquier secuencia de espacios en blanco por un solo espacio
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_chunks(text, chunk_size=1000):
    """
    Divide el texto en fragmentos (chunks) basados en número de palabras.
    Por defecto, cada fragmento tendrá aproximadamente 1000 palabras.
    """
    words = text.split()
    # Crea una lista de fragmentos uniendo las palabras en grupos de tamaño chunk_size
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Listas para almacenar los fragmentos y su metadata correspondiente
all_chunks = []
chunk_metadata = []

print("Iniciando procesamiento de los documentos...")

# Iteramos sobre los papers definidos en el catálogo JSON (catalog_data)
for paper in catalog_data["papers"]:
    # Construimos la ruta usando la variable universal 'papers_folder'
    file_path = os.path.join(papers_folder, paper["filename"])
    
    if os.path.exists(file_path):
        print(f"-> Procesando: {paper['title']}")
        
        # 1. Extracción
        raw_text = extract_text_from_pdf(file_path)
        
        # 2. Limpieza
        cleaned_txt = clean_text(raw_text)
        
        # 3. Fragmentación (Chunking)
        chunks = get_chunks(cleaned_txt)
        
        # 4. Organización de datos
        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            chunk_metadata.append({
                "paper_id": paper["id"],
                "title": paper["title"],
                "author": ", ".join(paper["authors"]) if isinstance(paper["authors"], list) else paper["authors"],
                "year": str(paper["year"]),
                "chunk_index": i
            })
    else:
        print(f"x Archivo no encontrado en la ruta: {file_path}")

print(f"\nProcesamiento completado.")
print(f"Total de fragmentos generados: {len(all_chunks)}")

Iniciando procesamiento de los documentos...
-> Procesando: From APEC to mega-regionals: the evolution of the Asia-Pacific trade architecture
-> Procesando: RCEP from the middle powers' Perspective
-> Procesando: ASEAN's inheritance: the regionalization of Southeast Asia, 1941-61
-> Procesando: Asean's inclusive regionalism: ambitious at three levels
-> Procesando: Investigating merchandise trade structure in the RCEP region from the perspective of regional integration
-> Procesando: Hedging via Institutions: ASEAN-led Multilateralism in the Age of the Indo-Pacific
-> Procesando: Assessing the Impact of the Regional Comprehensive Economic Partnership on ASEAN Trade
-> Procesando: Mega-Regional Trade Deals in the Asia-Pacific: Choosing Between the TPP and RCEP?
-> Procesando: RCEP: a strategic opportunity for multilateralism
-> Procesando: Towards a New ASEAN Regionalism: Navigating the Outlook on Indo-Pacific in Post-RCEP Beyond 2020
-> Procesando: Building ASEAN Identity Through Regio

### **BASE DE DATOS VECTORIAL (ChromaDB)**

In [6]:
# 1. Inicializar el cliente de ChromaDB
chroma_client = chromadb.Client()

# 2. Configurar el nombre de la colección
collection_name = "asean_research_papers"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)
# -------------------------------------------------------

# 3. Preparar los IDs únicos para cada fragmento
ids = [f"id_chunk_{i}" for i in range(len(all_chunks))]

print(f"Cargando {len(all_chunks)} fragmentos en la base de datos vectorial...")

# 4. Agregar los documentos a la colección
# Asegúrate de haber corrido la Celda 4 con la metadata corregida (author y year)
collection.add(
    documents=all_chunks,    
    metadatas=chunk_metadata, 
    ids=ids                  
)

print("------------------------------------------------------------------")
print("¡Éxito! Base de datos vectorial creada y poblada.")
print(f"Colección activa: '{collection_name}'")
print(f"Total de registros: {collection.count()}")
print("------------------------------------------------------------------")

Cargando 189 fragmentos en la base de datos vectorial...
------------------------------------------------------------------
¡Éxito! Base de datos vectorial creada y poblada.
Colección activa: 'asean_research_papers'
Total de registros: 189
------------------------------------------------------------------


### **SISTEMA DE CONSULTA (RAG)**

In [7]:
def query_research_copilot(user_query):
    """
    Función principal para consultar el asistente.
    1. Recupera los fragmentos más relevantes de ChromaDB.
    2. Construye un prompt con el contexto académico.
    3. Genera una respuesta precisa usando GPT.
    """
    
    # 1. RETRIEVAL (Recuperación)
    # Buscamos los 4 fragmentos más relevantes en la base de datos vectorial
    results = collection.query(query_texts=[user_query], n_results=4)
    retrieved_chunks = results['documents'][0]
    metadatas = results['metadatas'][0]
    
    
    # Unimos los fragmentos recuperados en un solo contexto, incluyendo referencias claras a su origen
    context = ""
    for i, text in enumerate(retrieved_chunks):
        # Usamos .get() para evitar errores si la llave no existe
        autor = metadatas[i].get('author', 'Anónimo')
        anio = metadatas[i].get('year', 's.f.')
        
        context += f"--- FUENTE PARA CITA: ({autor}, {anio}) ---\n{text}\n\n"
    
    # 2. PROMPT ENGINEERING (Diseño de la instrucción)
    prompt = f"""
    Eres un asistente de investigación especializado en Relaciones Internacionales y Asociación de Naciones del Sudeste Asiático (ASEAN).
    Tu objetivo es responder preguntas basadas únicamente en el contexto académico proporcionado a continuación.

    INSTRUCCIONES DE FORMATO:
    - Responde de forma exhaustiva pero directa.
    - DEBES incluir citas en formato APA (Autor, Año) dentro del texto cuando uses información de las fuentes.
    - Si la información no está en el contexto, indica que no puedes responder basándote en la literatura proporcionada.
    
    EJEMPLO DE FORMATO:
    Pregunta: ¿Qué es la centralidad de ASEAN?
    Respuesta: La centralidad se define como el rol conductor de la organización en la arquitectura regional (Solís & Wilson, 2017).

    CONTEXTO:
    {context}

    REGLAS DE RESPUESTA:
    1. Usa un tono académico, objetivo y profesional.
    2. Si la respuesta no se encuentra en el contexto, di: "Lo siento, la información disponible en los documentos no me permite responder a esa pregunta con precisión".
    3. Siempre que sea posible, menciona los títulos de los artículos que estás citando (esta información está en la metadata si decides extraerla, o dentro del mismo texto).

    PREGUNTA:
    {user_query}
    

    RESPUESTA:
    """

    # 3. GENERATION (Generación con OpenAI)
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", 
            messages=[
                {"role": "system", "content": "Eres un experto en el RCEP y la Centralidad de ASEAN."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3 # Temperatura baja para mayor precisión y menos "alucinación"
        )
        
        return response.choices[0].message.content
    
    except Exception as e:
        return f"Hubo un error al generar la respuesta: {e}"


print("--- RESEARCH COPILOT LISTO ---")
mi_pregunta = "¿Cómo se define la centralidad de ASEAN en el contexto de las negociaciones del RCEP?"
respuesta = query_research_copilot(mi_pregunta)

print(f"\nPREGUNTA: {mi_pregunta}")
print(f"\nRESPUESTA DEL ASISTENTE:\n{respuesta}")

--- RESEARCH COPILOT LISTO ---

PREGUNTA: ¿Cómo se define la centralidad de ASEAN en el contexto de las negociaciones del RCEP?

RESPUESTA DEL ASISTENTE:
La centralidad de ASEAN en el contexto de las negociaciones del RCEP se define como el papel fundamental que desempeña ASEAN en la arquitectura regional y su capacidad para liderar y gestionar procesos de cooperación económica en el este asiático. El RCEP, que es un acuerdo de asociación económica integral, fue propuesto y promovido por ASEAN, lo que subraya su rol como un actor clave en la integración económica regional (Kazushi Shimizu, 2021). 

La centralidad de ASEAN implica que los estados miembros priorizan el cumplimiento de sus compromisos regionales y trabajan de manera cohesiva para fortalecer su influencia en el ámbito internacional (Elaine Tan, 2022). Además, el RCEP no solo se considera un acuerdo de libre comercio, sino que también incorpora una agenda de cooperación que busca mejorar la capacidad de reforma económica y 